In [1]:
import numpy as np, pandas as pd
from m1_utils import *

queries, judg, corpus = load_all(); qrels = build_qrels(judg); pids = corpus["product_id"].to_numpy()
eval_q = pd.read_parquet(OUT / "eval_query_order.parquet")
P = pd.read_parquet(OUT / "clf_probs_eval.parquet"); assert (P["query_id"].values == eval_q["query_id"].values).all()
bm25_scores = np.load(OUT / "bm25_scores.npy", mmap_mode="r")
dense_scores = np.load(OUT / "dense_raw_scores.npy", mmap_mode="r")
G = pd.read_parquet(OUT / "dev_alpha_grid.parquet")
dev_rows = eval_q.index[eval_q["role"] == "dev"]

def cand_blend(i, K=100):
    b, d = np.asarray(bm25_scores[i]), np.asarray(dense_scores[i])
    cand = np.union1d(np.argpartition(-b, K)[:K], np.argpartition(-d, K)[:K])
    return cand, minmax(b[cand]), minmax(d[cand])
pre = {i: cand_blend(i) for i in dev_rows}

def run(alpha):                       # alpha: array indexed by eval_q row
    ranked = {}
    for i in dev_rows:
        cand, bn, dn = pre[i]
        s = alpha[i] * bn + (1 - alpha[i]) * dn
        ranked[eval_q.at[i, "query_id"]] = list(pids[cand[np.argsort(-s)[:10]]])
    return evaluate(ranked, qrels)["ndcg@10"]

n = len(eval_q)
res = {"fixed 0.4": run(np.full(n, 0.4)),
       "alpha=P(Exact) raw": run(P["pE_raw"].values),
       "alpha=P(Exact) temp-scaled": run(P["pE_T"].values)}
print(pd.DataFrame({k: [v.mean()] for k, v in res.items()}, index=["dev NDCG@10"]).round(4).T)
print(f"oracle per-query alpha: {G.max(axis=1).mean():.4f}")
print("\nmean alpha used:", {k: round(float(v), 3) for k, v in [("raw", P.loc[dev_rows, 'pE_raw'].mean()), ("scaled", P.loc[dev_rows, 'pE_T'].mean())]})

# does the predicted class separate alpha preferences? (G columns = alpha grid on dev)
g = G.loc[eval_q.loc[dev_rows, "query_id"]].copy(); g["pred"] = P.loc[dev_rows, "pred"].values
cols = ["a0.1", "a0.4", "a0.9"]
print("\nmean NDCG@10 by PREDICTED class at fixed alpha:")
print(g.groupby("pred")[cols].mean().round(3).assign(n=g["pred"].value_counts(), best=g.groupby("pred")[[c for c in G.columns]].mean().idxmax(axis=1)))

                            dev NDCG@10
fixed 0.4                        0.2950
alpha=P(Exact) raw               0.2526
alpha=P(Exact) temp-scaled       0.2652
oracle per-query alpha: 0.3669

mean alpha used: {'raw': 0.189, 'scaled': 0.206}

mean NDCG@10 by PREDICTED class at fixed alpha:
           a0.1   a0.4   a0.9    n  best
pred                                    
Exact     0.352  0.426  0.375   97  a0.4
Hybrid    0.274  0.294  0.249  196  a0.4
Semantic  0.203  0.235  0.216  207  a0.4


In [2]:
from scipy.stats import spearmanr
dens = {"C2 raw": np.asarray(dense_scores)[dev_rows.to_numpy()],
        "C3 avg": np.load(OUT / "dense_dev_C3avg_v1.npy"),
        "C4 avg": np.load(OUT / "dense_dev_C4avg_v1.npy")}
qids = eval_q.loc[dev_rows, "query_id"].tolist()
assert (np.load(OUT / "dense_dev_qids.npy") == np.array(qids)).all()
BM = np.asarray(bm25_scores)[dev_rows.to_numpy()]
pE = P.loc[dev_rows, "pE_T"].values; pred = P.loc[dev_rows, "pred"].values
n = len(qids); K = 100

pre2 = {}
for name, M in dens.items():
    L = []
    for i in range(n):
        b, d = BM[i], M[i]
        cand = np.union1d(np.argpartition(-b, K)[:K], np.argpartition(-d, K)[:K])
        L.append((cand, minmax(b[cand]), minmax(d[cand])))
    pre2[name] = L

def ndcg_vec(name, alpha):
    ranked = {}
    for i, q in enumerate(qids):
        cand, bn, dn = pre2[name][i]
        s = alpha[i] * bn + (1 - alpha[i]) * dn
        ranked[q] = list(pids[cand[np.argsort(-s)[:10]]])
    return evaluate(ranked, qrels)["ndcg@10"].values

def boot(a, b, nb=5000, seed=0):
    d = a - b; r = np.random.default_rng(seed)
    m = np.array([d[r.integers(0, len(d), len(d))].mean() for _ in range(nb)])
    return d.mean(), np.percentile(m, [2.5, 97.5])

grid = np.round(np.linspace(0, 1, 11), 1)
pairs = [(lo, hi) for lo in [0.1, 0.2, 0.3, 0.4, 0.5] for hi in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9] if hi >= lo]
folds = np.random.default_rng(0).permutation(n) % 5
def cv_pick(M):
    out = np.zeros(n)
    for f in range(5):
        tr = folds != f; j = M[tr].mean(0).argmax(); out[~tr] = M[~tr, j]
    return out

summary = []
for name in dens:
    Gm = np.column_stack([ndcg_vec(name, np.full(n, a)) for a in grid])                 # fixed-alpha grid
    Pm = np.column_stack([ndcg_vec(name, lo + (hi - lo) * pE) for lo, hi in pairs])      # mapped alpha grid
    formula = ndcg_vec(name, pE)                                                         # alpha = P(Exact) as proposed
    fixed_cv, mapped_cv = cv_pick(Gm), cv_pick(Pm)
    adv = Gm[:, 9] - Gm[:, 1]                                                            # BM25 advantage (alpha .9 vs .1)
    rho, pval = spearmanr(pE, adv)
    jb = Pm.mean(0).argmax()
    summary.append({"dense": name, "fixed (CV)": fixed_cv.mean(), "alpha=P(Exact)": formula.mean(),
                    "mapped lo+(hi-lo)P (CV)": mapped_cv.mean(), "oracle (grid)": Gm.max(1).mean(),
                    "best fixed a (full dev)": grid[Gm.mean(0).argmax()], "best map (full dev)": pairs[jb],
                    "rho(P_exact, BM25 advantage)": round(rho, 3), "p": round(pval, 3)})
    m1, c1 = boot(mapped_cv, fixed_cv); m2, c2 = boot(formula, fixed_cv)
    print(f"[{name}] mapped(CV)-fixed(CV): {m1:+.4f} CI [{c1[0]:+.4f},{c1[1]:+.4f}] | formula-fixed(CV): {m2:+.4f} CI [{c2[0]:+.4f},{c2[1]:+.4f}]")

print(); print(pd.DataFrame(summary).set_index("dense").round(4).T.to_string())

g = pd.DataFrame({"pred": pred, "a0.1": ndcg_vec("C4 avg", np.full(n, .1)), "a0.4": ndcg_vec("C4 avg", np.full(n, .4)), "a0.9": ndcg_vec("C4 avg", np.full(n, .9))})
print("\nC4 avg: mean NDCG@10 by predicted class"); print(g.groupby("pred").mean().round(3).assign(n=g["pred"].value_counts()))

[C2 raw] mapped(CV)-fixed(CV): -0.0014 CI [-0.0040,+0.0011] | formula-fixed(CV): -0.0299 CI [-0.0414,-0.0179]
[C3 avg] mapped(CV)-fixed(CV): +0.0025 CI [-0.0006,+0.0056] | formula-fixed(CV): -0.0338 CI [-0.0458,-0.0222]
[C4 avg] mapped(CV)-fixed(CV): +0.0000 CI [+0.0000,+0.0000] | formula-fixed(CV): -0.0378 CI [-0.0508,-0.0251]

dense                             C2 raw      C3 avg      C4 avg
fixed (CV)                         0.295      0.3137      0.3168
alpha=P(Exact)                    0.2652      0.2799       0.279
mapped lo+(hi-lo)P (CV)           0.2936      0.3162      0.3168
oracle (grid)                     0.3669      0.3855      0.3899
best fixed a (full dev)              0.4         0.4         0.4
best map (full dev)           (0.4, 0.4)  (0.3, 0.4)  (0.4, 0.4)
rho(P_exact, BM25 advantage)      -0.007      -0.083      -0.099
p                                  0.881       0.063       0.027

C4 avg: mean NDCG@10 by predicted class
           a0.1   a0.4   a0.9    n
pred    

In [3]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import torch

clf_df = pd.concat([pd.read_csv(OUT / f"classifier_{n}.csv") for n in ["train", "val"]]).reset_index(drop=True)
print(len(clf_df), clf_df["intent"].value_counts().to_dict())

bm25 = BM25Okapi([tokenize(t) for t in corpus["doc_text"]])
BMc = np.vstack([bm25.get_scores(tokenize(t)) for t in clf_df["query"]]).astype("float32")
st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu"); st.max_seq_length = 256
doc_emb = np.load(OUT / "doc_emb_minilm.npy")
Dc = st.encode(clf_df["query"].tolist(), batch_size=64, normalize_embeddings=True, convert_to_numpy=True).astype("float32") @ doc_emb.T

K = 100; grid = np.round(np.linspace(0, 1, 11), 1)
def ndcg_grid(BMm, Dm, ids):
    out = np.zeros((len(ids), len(grid)))
    for i, q in enumerate(ids):
        b, d = BMm[i], Dm[i]
        cand = np.union1d(np.argpartition(-b, K)[:K], np.argpartition(-d, K)[:K])
        bn, dn = minmax(b[cand]), minmax(d[cand])
        for j, a in enumerate(grid):
            s = a * bn + (1 - a) * dn
            out[i, j] = evaluate({q: list(pids[cand[np.argsort(-s)[:10]]])}, qrels).iloc[0, 0]
    return out
Gc = ndcg_grid(BMc, Dc, clf_df["query_id"].tolist())

tab = pd.DataFrame(Gc, columns=grid).assign(intent=clf_df["intent"].values).groupby("intent").mean().round(3)
tab["n"] = clf_df["intent"].value_counts(); print("\nmean NDCG@10 by HUMAN intent and alpha (alpha=1 is BM25, 0 is dense):"); print(tab.to_string())
print("best alpha per class:", tab.drop(columns="n").idxmax(axis=1).to_dict())

isE = (clf_df["intent"] == "Exact").values.astype(float)
adv = Gc[:, 9] - Gc[:, 1]
rho, p = spearmanr(isE, adv); print(f"\nrho(human Exact, BM25 advantage) = {rho:.3f} (p={p:.3f})")
print("mean BM25 advantage (a=.9 minus a=.1) by class:", pd.Series(adv).groupby(clf_df["intent"].values).mean().round(3).to_dict())

fixed = Gc[:, list(grid).index(0.4)]
byclass = np.zeros(len(clf_df))
for c in clf_df["intent"].unique():
    m = (clf_df["intent"] == c).values; byclass[m] = Gc[m][:, Gc[m].mean(0).argmax()]
m_, ci = boot(byclass, fixed)
print(f"\nceiling for ANY intent-class alpha (in-sample, perfect labels) minus fixed 0.4: {m_:+.4f} CI [{ci[0]:+.4f},{ci[1]:+.4f}]")
print(f"fixed 0.4: {fixed.mean():.4f} | per-class best: {byclass.mean():.4f} | oracle per-query: {Gc.max(1).mean():.4f}")

469 {'Hybrid': 184, 'Semantic': 173, 'Exact': 112}


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


mean NDCG@10 by HUMAN intent and alpha (alpha=1 is BM25, 0 is dense):
            0.0    0.1    0.2    0.3    0.4    0.5    0.6    0.7    0.8    0.9    1.0    n
intent                                                                                    
Exact     0.431  0.467  0.478  0.484  0.494  0.484  0.477  0.471  0.464  0.455  0.441  112
Hybrid    0.312  0.329  0.340  0.333  0.331  0.325  0.308  0.290  0.280  0.264  0.252  184
Semantic  0.122  0.136  0.148  0.153  0.157  0.159  0.160  0.160  0.156  0.159  0.149  173
best alpha per class: {'Exact': 0.4, 'Hybrid': 0.2, 'Semantic': 0.6}

rho(human Exact, BM25 advantage) = 0.008 (p=0.865)
mean BM25 advantage (a=.9 minus a=.1) by class: {'Exact': -0.012, 'Hybrid': -0.065, 'Semantic': 0.023}

ceiling for ANY intent-class alpha (in-sample, perfect labels) minus fixed 0.4: +0.0050 CI [-0.0011,+0.0110]
fixed 0.4: 0.3055 | per-class best: 0.3104 | oracle per-query: 0.3780
